# Customer Churn Analysis & Retention Strategy

## Phase 1: Business framing and data audit

Run this notebook from top to bottom using the project environment. Each code cell answers one small question. We inspect the raw data without cleaning it.

If you know SQL: a pandas **DataFrame** is similar to a table, a **Series** is one column, a Boolean condition works like **WHERE**, and **groupby** works like **GROUP BY**. A **Boolean mask** is a column of True/False values used to select rows.

Business decisions and definitions: [analysis brief](../outputs/reports/analysis_brief.md). Audit evidence: [analysis log](../outputs/reports/analysis_log.md). No customer-level records are displayed.

### Step 2.1: Load the file and record its identity

First import the libraries. pandas handles tables; NumPy provides the finite-number check. The remaining imports record the file location, environment, time, and checksum. A checksum identifies the exact file and helps us confirm that it stays unchanged.

#### Imports and display settings

The standard-library imports handle paths, dates, CSV headers, versions, and checksums. pandas handles tables; NumPy checks numbers. display() shows a readable table. set_option() changes display only, not data.

In [1]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
import csv
import hashlib
import platform

import pandas as pd
import numpy as np
from IPython.display import display

# Show every column in the small audit tables.
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

Find the project folder by checking the current folder, then its parents. This supports running from the repository root or a folder inside it, including notebooks/.

#### Locate the project folder

Path.cwd() returns the current working folder. resolve() makes its path absolute. The / operator joins path parts. while repeats until the input exists; parent moves up one folder. raise stops with a clear error if the file cannot be found.

In [2]:
project_root = Path.cwd().resolve()

while not (project_root / "datasets/TelecomChurnData.csv").is_file():
    parent_folder = project_root.parent
    if parent_folder == project_root:
        raise FileNotFoundError("Run this notebook from inside CustomerChurnPrediction.")
    project_root = parent_folder

#### Name the files and record the checksum

Path objects hold file locations. read_bytes() reads the original bytes; SHA-256 turns them into a file fingerprint. hexdigest() makes that fingerprint printable. We compare it again at the end.

In [3]:
raw_data_path = project_root / "datasets/TelecomChurnData.csv"
dictionary_path = project_root / "outputs/reports/data_dictionary.csv"

source_hash = hashlib.sha256(raw_data_path.read_bytes()).hexdigest()
print("Input:", raw_data_path.name)
print("SHA-256:", source_hash)

Input: TelecomChurnData.csv
SHA-256: 88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a


Read two views of the same CSV. The normal view lets pandas infer types and missing values. The text view preserves blank strings exactly as supplied. This matters because spaces in TotalCharges are not detected by a normal null check. Copies let us confirm later that neither view was changed.

#### Read the two raw views

read_csv() creates a DataFrame. dtype=str keeps every value as text; keep_default_na=False and na_filter=False preserve blank/NA-like text. UTF-8 is the text encoding. Neither view is cleaned.

In [4]:
raw_df = pd.read_csv(raw_data_path, encoding="utf-8")
literal_df = pd.read_csv(
    raw_data_path,
    encoding="utf-8",
    dtype=str,
    keep_default_na=False,
    na_filter=False
)

#### Save comparison copies and execution details

copy() keeps a separate table for the final equality check. __version__ records a library version. ZoneInfo chooses the local timezone, and isoformat() formats the timestamp. len() counts rows or column names.

In [5]:
raw_snapshot = raw_df.copy()
literal_snapshot = literal_df.copy()

print("Audit time:", datetime.now(ZoneInfo("Asia/Calcutta")).isoformat())
print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Rows:", len(raw_df))
print("Columns:", len(raw_df.columns))
print("Loading: UTF-8, comma delimiter, first row as header.")
print("Text view: dtype=str, keep_default_na=False, na_filter=False.")

Audit time: 2026-11-06T22:40:29.313134+05:30
Python: 3.14.6
pandas: 3.1.1
NumPy: 2.5.4
Rows: 7043
Columns: 21
Loading: UTF-8, comma delimiter, first row as header.
Text view: dtype=str, keep_default_na=False, na_filter=False.


### Step 2.2: Check the schema and identifiers

The existing dictionary lists the expected fields. csv.reader reads the original header so that duplicate column names cannot be hidden by pandas renaming them. Simple loops compare the expected and actual names.

#### Read expected fields and the original header

Selecting ["field"] returns one column; tolist() turns it into a Python list. with closes the file automatically. csv.reader() reads CSV rows; next(reader) reads its first row, the header.

In [6]:
dictionary_seed = pd.read_csv(dictionary_path, keep_default_na=False)
expected_fields = dictionary_seed["field"].tolist()

with raw_data_path.open(encoding="utf-8", newline="") as source_file:
    reader = csv.reader(source_file)
    header = next(reader)

#### Compare actual and expected field names

An empty list starts as []. for visits each name; in checks membership; append() adds an item. header.count(field) counts repeated header names. This is similar to comparing two SQL schema lists.

In [7]:
missing_fields = []
unexpected_fields = []
duplicate_headers = []

for field in expected_fields:
    if field not in header:
        missing_fields.append(field)

for field in header:
    if field not in expected_fields:
        unexpected_fields.append(field)
    if header.count(field) > 1 and field not in duplicate_headers:
        duplicate_headers.append(field)

#### Review schema differences and column types

dtypes reports pandas types. to_frame() turns a Series into a one-column table; its argument supplies the column heading. Empty mismatch lists mean no schema differences were found.

In [8]:
print("Missing fields:", missing_fields)
print("Unexpected fields:", unexpected_fields)
print("Duplicate header names:", duplicate_headers)
display(raw_df.dtypes.to_frame("observed_type"))

Missing fields: []
Unexpected fields: []
Duplicate header names: []
                 observed_type
customerID                 str
gender                     str
SeniorCitizen            int64
Partner                    str
Dependents                 str
tenure                   int64
PhoneService               str
MultipleLines              str
InternetService            str
OnlineSecurity             str
OnlineBackup               str
DeviceProtection           str
TechSupport                str
StreamingTV                str
StreamingMovies            str
Contract                   str
PaperlessBilling           str
PaymentMethod              str
MonthlyCharges         float64
TotalCharges               str
Churn                      str


Check missing IDs, repeated IDs, and exact duplicate rows separately. duplicated(keep=False) flags **every** row in a duplicate group; duplicated() flags only excess copies. None of these checks removes data.

#### Prepare ID conditions

str.strip() removes outer whitespace in a separate view. A comparison returns a True/False Series, called a Boolean mask. & means AND. duplicated(keep=False) marks every participant in a repeated-ID group.

In [9]:
ids = literal_df["customerID"]
trimmed_ids = ids.str.strip()

valid_id = trimmed_ids != ""
repeated_id = valid_id & ids.duplicated(keep=False)

#### Count missing or blank identifiers

A Python dictionary stores named results as key/value pairs. Boolean sum() counts True values. isna() checks pandas missing values; nunique() counts distinct values, like COUNT(DISTINCT ...). ids[valid_id] filters rows like WHERE.

In [10]:
identifier_audit = {}
identifier_audit["inferred_null_ids"] = raw_df["customerID"].isna().sum()
identifier_audit["empty_ids"] = (ids == "").sum()
identifier_audit["whitespace_only_ids"] = ((ids != "") & (trimmed_ids == "")).sum()
identifier_audit["unique_nonblank_ids"] = ids[valid_id].nunique()

#### Count duplicate rows and repeated IDs

duplicated() marks excess copies; keep=False includes every participating row. These are different counts. The checks only count issues; nothing is dropped or changed.

In [11]:
identifier_audit["exact_duplicate_excess_rows"] = literal_df.duplicated().sum()
identifier_audit["exact_duplicate_participating_rows"] = literal_df.duplicated(keep=False).sum()
identifier_audit["repeated_id_rows"] = repeated_id.sum()
identifier_audit["repeated_id_groups"] = ids[repeated_id].nunique()
identifier_audit["ids_with_outer_whitespace"] = (ids != trimmed_ids).sum()

A repeated ID might have identical records or conflicting attributes. After SELECT DISTINCT across complete rows, GROUP BY customerID and count the distinct records. More than one record means a conflict. Also check whether trimming and lowercasing would merge different IDs; this is only a diagnostic, not a change to IDs.

#### Find conflicting records for the same ID

drop_duplicates() here creates a temporary SELECT DISTINCT view. groupby("customerID").size() counts its rows per ID. Filtering counts greater than one finds conflicts; .index returns their ID labels. isin() checks membership in that list.

In [12]:
distinct_records = literal_df[valid_id].drop_duplicates()
id_record_counts = distinct_records.groupby("customerID").size()
conflict_ids = id_record_counts[id_record_counts > 1].index

identifier_audit["conflicting_id_groups"] = len(conflict_ids)
identifier_audit["conflicting_id_rows"] = ids.isin(conflict_ids).sum()

#### Check ID spelling collisions and show results

casefold() makes text comparable without case differences. A reduction in distinct count would mean different original IDs collapse after normalisation. pd.Series() turns the result dictionary into a labelled column. Original IDs are preserved.

In [13]:
original_id_count = ids[valid_id].nunique()
normalised_ids = trimmed_ids[valid_id].str.casefold()
identifier_audit["distinct_ids_collapsing_after_strip_casefold"] = (
    original_id_count - normalised_ids.nunique()
)

identifier_audit = pd.Series(identifier_audit, name="count")
display(identifier_audit.to_frame())

                                              count
inferred_null_ids                                 0
empty_ids                                         0
whitespace_only_ids                               0
unique_nonblank_ids                            7043
exact_duplicate_excess_rows                       0
exact_duplicate_participating_rows                0
repeated_id_rows                                  0
repeated_id_groups                                0
ids_with_outer_whitespace                         0
conflicting_id_groups                             0
conflicting_id_rows                               0
distinct_ids_collapsing_after_strip_casefold      0


### Step 2.3: Check missing values and categories

For each field, count inferred nulls, empty strings, whitespace-only strings, and text that pandas interprets as missing. These measures can overlap and should not be added together. strip() creates a diagnostic view without changing the original values.

#### Start the field summary

astype(str) turns dtype names into text for display. Assigning summary["column"] adds a column. pandas aligns Series by their index labels, so each source field receives its own counts.

In [14]:
# Each assignment adds one column to this small summary table.
field_profile = raw_df.dtypes.astype(str).to_frame("observed_dtype")
field_profile["inferred_nulls"] = raw_df.isna().sum()
field_profile["distinct_literal_values"] = literal_df.nunique()

#### Count empty, whitespace, and NA-like text

!= means not equal; ~ means NOT for a pandas mask. .loc[field, column] addresses a summary cell by its row and column labels. Each sum() counts one condition; the counts should not be added because they may overlap.

In [15]:
for field in header:
    text = literal_df[field]
    trimmed = text.str.strip()
    empty = text == ""
    spaces_only = (text != "") & (trimmed == "")
    nonblank_na = raw_df[field].isna() & (trimmed != "")

    field_profile.loc[field, "empty_strings"] = empty.sum()
    field_profile.loc[field, "whitespace_only"] = spaces_only.sum()
    field_profile.loc[field, "nonblank_na_tokens"] = nonblank_na.sum()
    field_profile.loc[field, "outer_whitespace"] = (text != trimmed).sum()

#### Display the completeness summary

index.name labels the row index for readability. display() shows the completed summary; it does not export it.

In [16]:
field_profile.index.name = "field"
display(field_profile)

                 observed_dtype  inferred_nulls  distinct_literal_values  \
field                                                                      
customerID                  str               0                     7043   
gender                      str               0                        2   
SeniorCitizen             int64               0                        2   
Partner                     str               0                        2   
Dependents                  str               0                        2   
tenure                    int64               0                       73   
PhoneService                str               0                        2   
MultipleLines               str               0                        3   
InternetService             str               0                        3   
OnlineSecurity              str               0                        3   
OnlineBackup                str               0                        3   
DeviceProtec

value_counts() is similar to GROUP BY value, COUNT(*). Keep valid service states such as No internet service. We exclude identifiers and numeric measures from category frequency tables.

#### Choose the category fields

numeric_fields is a list of measurement columns. The loop selects the other fields except customerID. and combines ordinary Python conditions; & combines pandas Boolean masks.

In [17]:
numeric_fields = ["tenure", "MonthlyCharges", "TotalCharges"]
category_fields = []

for field in header:
    if field != "customerID" and field not in numeric_fields:
        category_fields.append(field)

#### Count each category directly

value_counts() performs GROUP BY value, COUNT(*). counts is a Series; category_counts is a dictionary that keeps one Series per field. items are not recoded, and no-service states remain valid categories.

In [18]:
# One simple GROUP BY count per category field.
category_counts = {}
for field in category_fields:
    counts = literal_df[field].value_counts()
    category_counts[field] = counts
    print(field)
    display(counts.to_frame("rows"))

gender
        rows
gender      
Male    3555
Female  3488
SeniorCitizen
               rows
SeniorCitizen      
0              5901
1              1142
Partner
         rows
Partner      
No       3641
Yes      3402
Dependents
            rows
Dependents      
No          4933
Yes         2110
PhoneService
              rows
PhoneService      
Yes           6361
No             682
MultipleLines
                  rows
MultipleLines         
No                3390
Yes               2971
No phone service   682
InternetService
                 rows
InternetService      
Fiber optic      3096
DSL              2421
No               1526
OnlineSecurity
                     rows
OnlineSecurity           
No                   3498
Yes                  2019
No internet service  1526
OnlineBackup
                     rows
OnlineBackup             
No                   3088
Yes                  2429
No internet service  1526
DeviceProtection
                     rows
DeviceProtection         
No 

Look for spelling variants after trimming and case-insensitive comparison. Start with each distinct spelling, then count how many become the same normalised value. We only inspect them; no category is changed.

#### Check for multiple spellings of the same category

drop_duplicates() returns distinct raw spellings. After strip() and casefold(), value_counts() counts spellings that would collapse to one category. pd.DataFrame(index=...) starts an empty summary with named rows; .loc fills its cells.

In [19]:
category_audit = pd.DataFrame(index=category_fields)

for field in category_fields:
    text = literal_df[field]
    unique_spellings = text.drop_duplicates()
    normalised_spellings = unique_spellings.str.strip().str.casefold()
    spelling_counts = normalised_spellings.value_counts()

    category_audit.loc[field, "outer_whitespace_rows"] = (text != text.str.strip()).sum()
    category_audit.loc[field, "normalisation_collision_groups"] = (spelling_counts > 1).sum()

#### Review category spelling checks

Each summary row represents one field. A zero collision count means its observed spellings do not merge under the diagnostic normalisation.

In [20]:
category_audit.index.name = "field"
display(category_audit)

                  outer_whitespace_rows  normalisation_collision_groups
field                                                                  
gender                              0.0                             0.0
SeniorCitizen                       0.0                             0.0
Partner                             0.0                             0.0
Dependents                          0.0                             0.0
PhoneService                        0.0                             0.0
MultipleLines                       0.0                             0.0
InternetService                     0.0                             0.0
OnlineSecurity                      0.0                             0.0
OnlineBackup                        0.0                             0.0
DeviceProtection                    0.0                             0.0
TechSupport                         0.0                             0.0
StreamingTV                         0.0                         

#### Check the senior-citizen flag vocabulary

isin(["0", "1"]) accepts only those exact text codes. ~ reverses the condition to find unexpected codes. This confirms observed coding, not the source definition of senior citizenship.

In [21]:
unexpected_senior_codes = ~literal_df["SeniorCitizen"].isin(["0", "1"])
print("Unexpected SeniorCitizen codes:", unexpected_senior_codes.sum())

Unexpected SeniorCitizen codes: 0


### Step 2.4: Check numeric fields

to_numeric(errors="coerce") turns unparseable values into missing numbers **in a separate diagnostic table**. We count conversion problems before deciding any treatment. None of these conversions cleans the raw file.

#### Convert numeric diagnostic values

pd.to_numeric(errors="coerce") converts parseable text to numbers and makes unparseable text missing. astype(float) uses a consistent numeric type. index=literal_df.index preserves row alignment. These operations populate a separate diagnostic table; raw values remain unchanged.

In [22]:
numeric_values = pd.DataFrame(index=literal_df.index)

for field in numeric_fields:
    trimmed_text = literal_df[field].str.strip()
    numeric_values[field] = pd.to_numeric(trimmed_text, errors="coerce")
    numeric_values[field] = numeric_values[field].astype(float)

numeric_audit = pd.DataFrame(index=numeric_fields)
special_tokens = [
    "nan", "+nan", "-nan", "inf", "+inf", "-inf",
    "infinity", "+infinity", "-infinity"
]

A finite number excludes infinity and NaN. Count blanks separately from malformed nonblank text and explicit non-finite tokens. Fractional charges are normal; fractional tenure is a review flag while its unit is unverified.

#### Count blank, malformed, and non-finite numbers

np.isfinite() is True for ordinary finite numbers and False for NaN or infinity. isna()/notna() test missingness. special_tokens distinguishes explicit non-finite spellings from other unparseable text. Parentheses keep AND/OR conditions unambiguous.

In [23]:
for field in numeric_fields:
    text = literal_df[field].str.strip()
    values = numeric_values[field]
    blank = text == ""
    special_token = text.str.lower().isin(special_tokens)
    finite = np.isfinite(values)
    parse_failure = ~blank & values.isna() & ~special_token
    nonfinite = ~blank & (special_token | (values.notna() & ~finite))
    finite_values = values[finite]

    numeric_audit.loc[field, "blank_rows"] = blank.sum()
    numeric_audit.loc[field, "nonblank_parse_failures"] = parse_failure.sum()
    numeric_audit.loc[field, "nonfinite_token_rows"] = nonfinite.sum()

#### Check finite ranges and whole-number tenure

Filtering values[finite] excludes missing and infinite numbers from ranges. % 1 gives a remainder: a nonzero result flags a fractional number. Fractional charges are normal; fractional tenure needs review while units are unverified.

In [24]:
for field in numeric_fields:
    values = numeric_values[field]
    finite = np.isfinite(values)
    finite_values = values[finite]

    numeric_audit.loc[field, "negative_rows"] = (finite & (values < 0)).sum()
    numeric_audit.loc[field, "fractional_rows"] = (finite & (values % 1 != 0)).sum()
    numeric_audit.loc[field, "finite_count"] = finite.sum()
    numeric_audit.loc[field, "minimum"] = finite_values.min()
    numeric_audit.loc[field, "maximum"] = finite_values.max()

Use the usual 1.5-IQR rule to flag unusually low or high values. The IQR is the distance between the 25th and 75th percentiles. These are review flags, not proof of errors, and they never trigger automatic removal. Source-supported maximum values are still unknown.

#### Calculate distribution-based review fences

quantile(0.25) and quantile(0.75) find the lower and upper quartiles. Their difference is the IQR. Values beyond the 1.5-IQR fences are review flags, not automatic errors or exclusions.

In [25]:
for field in numeric_fields:
    values = numeric_values[field]
    finite = np.isfinite(values)
    finite_values = values[finite]

    q1 = finite_values.quantile(0.25)
    q3 = finite_values.quantile(0.75)
    iqr = q3 - q1
    lower_fence = q1 - 1.5 * iqr
    upper_fence = q3 + 1.5 * iqr
    outside_fences = (values < lower_fence) | (values > upper_fence)

    numeric_audit.loc[field, "iqr_lower_fence"] = lower_fence
    numeric_audit.loc[field, "iqr_upper_fence"] = upper_fence
    numeric_audit.loc[field, "iqr_review_rows"] = (finite & outside_fences).sum()

#### Review the numeric audit

This displays the counts and ranges calculated in the preceding cells. No source-supported maximum values are known.

In [26]:
numeric_audit.index.name = "field"
display(numeric_audit)

                blank_rows  nonblank_parse_failures  nonfinite_token_rows  \
field                                                                       
tenure                 0.0                      0.0                   0.0   
MonthlyCharges         0.0                      0.0                   0.0   
TotalCharges          11.0                      0.0                   0.0   

                negative_rows  fractional_rows  finite_count  minimum  \
field                                                                   
tenure                    0.0              0.0        7043.0     0.00   
MonthlyCharges            0.0           6651.0        7043.0    18.25   
TotalCharges              0.0           6708.0        7032.0    18.80   

                maximum  iqr_lower_fence  iqr_upper_fence  iqr_review_rows  
field                                                                       
tenure            72.00        -60.00000        124.00000              0.0  
MonthlyCharges   

Investigate blank TotalCharges by tenure. crosstab() counts combinations, like a SQL GROUP BY arranged as a pivot. Display only tenure groups containing blanks, then check whether any blanks occur outside zero tenure. The cause remains unknown; do not assume the correct value is zero.

#### Locate tenure groups containing blank totals

crosstab() counts combinations like a pivoted GROUP BY. rename(columns={...}) changes output headings using a dictionary. The Boolean filter displays only groups needing investigation; the complete cross-tab remains in memory.

In [27]:
total_blank = literal_df["TotalCharges"].str.strip() == ""
zero_tenure = numeric_values["tenure"] == 0

total_blank_by_tenure = pd.crosstab(numeric_values["tenure"], total_blank)
total_blank_by_tenure = total_blank_by_tenure.rename(columns={False: "nonblank_TotalCharges", True: "blank_TotalCharges"})

# Focus the output on tenure groups with missing totals.
groups_with_blanks = total_blank_by_tenure["blank_TotalCharges"] > 0
display(total_blank_by_tenure[groups_with_blanks])

TotalCharges  nonblank_TotalCharges  blank_TotalCharges
tenure                                                 
0.0                               0                  11


#### Check the blank-total relationship

& counts records meeting both conditions; ~zero_tenure means all other tenure values. Blanks occurring at zero tenure do not prove that the correct total is zero.

In [28]:
print("Zero-tenure rows:", zero_tenure.sum())
print("Blank totals at zero tenure:", (total_blank & zero_tenure).sum())
print("Blank totals at other tenure:", (total_blank & ~zero_tenure).sum())

Zero-tenure rows: 11
Blank totals at zero tenure: 11
Blank totals at other tenure: 0


#### Compare totals with monthly charges

Two Series are compared row by row using their indexes. sum() counts True comparisons. Billing timing is unknown, so the result is only a diagnostic flag.

In [29]:
total_below_monthly = numeric_values["TotalCharges"] < numeric_values["MonthlyCharges"]
print("Totals below monthly charges:", total_below_monthly.sum())
print("Billing units and timing are unverified; this comparison is only a review flag.")

Totals below monthly charges: 0
Billing units and timing are unverified; this comparison is only a review flag.


### Step 2.5: Check service relationships

Check both the expected vocabulary and whether no-service states agree. Start with phone service. & means AND, | means OR, and ~ means NOT when combining pandas Boolean masks; put each comparison in parentheses.

#### Check allowed phone-service values

The service_checks dictionary stores one Boolean mask per check. isin() accepts the listed vocabulary; ~ selects unexpected values.

In [30]:
service_checks = {}
phone = literal_df["PhoneService"]
lines = literal_df["MultipleLines"]

service_checks["unexpected_PhoneService"] = ~phone.isin(["Yes", "No"])
service_checks["unexpected_MultipleLines"] = ~lines.isin(["Yes", "No", "No phone service"])

#### Check phone-service dependencies

The first condition catches a no-phone record with an inconsistent line state. The second catches an active phone record marked as no service. | combines the two populations without double-counting rows in that mask.

In [31]:
no_phone_but_lines_recorded = (phone == "No") & (lines != "No phone service")
has_phone_but_no_service_state = (phone == "Yes") & (lines == "No phone service")
service_checks["phone_dependency_mismatch"] = (
    no_phone_but_lines_recorded | has_phone_but_no_service_state
)

Repeat the same check for the six services that depend on internet service. Keep No internet service as a valid value. These are schema-based checks pending source definitions.

#### Set up internet-service conditions

The field list names six services that depend on internet access. Build has_internet and no_internet masks once, then reuse them in the following loop.

In [32]:
internet = literal_df["InternetService"]
internet_fields = [
    "OnlineSecurity", "OnlineBackup", "DeviceProtection",
    "TechSupport", "StreamingTV", "StreamingMovies"
]
service_checks["unexpected_InternetService"] = ~internet.isin(["DSL", "Fiber optic", "No"])
no_internet = internet == "No"
has_internet = internet.isin(["DSL", "Fiber optic"])

#### Check each internet-dependent service

String + joins a check prefix and field name to create a readable dictionary key. Each loop checks the allowed values and both dependency directions for that field. No internet service is a legitimate state.

In [33]:
for field in internet_fields:
    service = literal_df[field]
    allowed_values = ["Yes", "No", "No internet service"]
    service_checks["unexpected_" + field] = ~service.isin(allowed_values)

    wrong_no_service_state = no_internet & (service != "No internet service")
    wrong_active_service_state = has_internet & (service == "No internet service")
    service_checks["internet_dependency_" + field] = (
        wrong_no_service_state | wrong_active_service_state
    )

#### Summarise service flags

DataFrame(service_checks) makes one column per Boolean check. sum() counts failures by column. any(axis=1) works across columns within each row to find records failing at least one check; summing it counts the union.

In [34]:
# Each column is one check. Sum counts failures; any finds affected rows.
service_flags = pd.DataFrame(service_checks)
service_audit = service_flags.sum()
service_audit.name = "flagged_rows"
display(service_audit.to_frame())
print("Rows with any service flag:", service_flags.any(axis=1).sum())

                                      flagged_rows
unexpected_PhoneService                          0
unexpected_MultipleLines                         0
phone_dependency_mismatch                        0
unexpected_InternetService                       0
unexpected_OnlineSecurity                        0
internet_dependency_OnlineSecurity               0
unexpected_OnlineBackup                          0
internet_dependency_OnlineBackup                 0
unexpected_DeviceProtection                      0
internet_dependency_DeviceProtection             0
unexpected_TechSupport                           0
internet_dependency_TechSupport                  0
unexpected_StreamingTV                           0
internet_dependency_StreamingTV                  0
unexpected_StreamingMovies                       0
internet_dependency_StreamingMovies              0
Rows with any service flag: 0


### Step 2.6: Check labels and calculate the baseline

Only exact Yes and No labels are accepted. Missing or unexpected labels are not treated as No. Show their counts before defining eligibility.

#### Inspect the recorded churn labels

value_counts() shows exact observed labels. We do not infer churn timing or silently replace unknown labels.

In [35]:
labels = literal_df["Churn"]
display(labels.value_counts().to_frame("rows"))

       rows
Churn      
No     5174
Yes    1869


#### Count invalid outcome labels

pd.Series({...}) creates a labelled summary from a dictionary. A valid_label mask accepts exact Yes/No values; the other masks distinguish empty, whitespace-only, and unexpected nonblank labels.

In [36]:
blank_label = labels.str.strip() == ""
valid_label = labels.isin(["Yes", "No"])

outcome_audit = pd.Series({
    "empty_labels": (labels == "").sum(),
    "whitespace_only_labels": ((labels != "") & blank_label).sum(),
    "unexpected_nonblank_labels": (~blank_label & ~valid_label).sum()
}, name="rows")
display(outcome_audit.to_frame())

                            rows
empty_labels                   0
whitespace_only_labels         0
unexpected_nonblank_labels     0


Eligibility works like WHERE all checks pass: ID present, no outer whitespace, no repeated ID, and a valid label. Separate exclusion reasons may overlap, so count their union for the total excluded population.

#### Determine the eligible population

Each exclusion-reason column is a True/False condition. any(axis=1) flags a row with at least one reason; ~excluded selects rows with none. This is equivalent to WHERE every eligibility check passes.

In [37]:
exclusion_reasons = pd.DataFrame()
exclusion_reasons["blank_id"] = ~valid_id
exclusion_reasons["id_outer_whitespace"] = ids != ids.str.strip()
exclusion_reasons["repeated_id"] = repeated_id
exclusion_reasons["invalid_label"] = ~valid_label

excluded = exclusion_reasons.any(axis=1)
eligible = ~excluded
display(exclusion_reasons.sum().to_frame("rows"))

                     rows
blank_id                0
id_outer_whitespace     0
repeated_id             0
invalid_label           0


#### Count eligible, churned, and retained records

Combine eligible with the outcome mask to count only eligible customers. int() converts the library count into a plain Python integer. No charge-completeness condition changes these denominators.

In [38]:
eligible_count = int(eligible.sum())
churn_count = int((eligible & (labels == "Yes")).sum())
retained_count = int((eligible & (labels == "No")).sum())

#### Calculate recorded churn prevalence

Check the denominator before dividing. None represents an unavailable result when no eligible customers exist. Multiplying by 100 expresses the ratio as a percentage.

In [39]:
if eligible_count > 0:
    churn_rate_pct = 100 * churn_count / eligible_count
else:
    churn_rate_pct = None

#### Display baseline counts and the rate

round(value, 2) rounds only the displayed percentage. is not None checks whether a result is available. Underlying calculations keep their precision.

In [40]:
print("Eligible unique customers:", eligible_count)
print("Excluded rows:", excluded.sum())
print("Churned:", churn_count)
print("Retained:", retained_count)
if churn_rate_pct is not None:
    print("Recorded churn (%):", round(churn_rate_pct, 2))
else:
    print("Recorded churn: unavailable; no eligible customers.")

Eligible unique customers: 7043
Excluded rows: 0
Churned: 1869
Retained: 5174
Recorded churn (%): 26.54


#### Reconcile eligibility totals

assert stops if a condition is false. .loc[eligible, "customerID"] selects eligible IDs; is_unique checks they appear once. Both reconciliation equations must hold before interpreting the baseline.

In [41]:
# These counts describe recorded labels, not monthly or future churn.
assert literal_df.loc[eligible, "customerID"].is_unique
assert eligible_count + excluded.sum() == len(literal_df)
assert churn_count + retained_count == eligible_count

### Step 2.7: Read the dictionary and confirm preservation

The reviewed [data dictionary](../outputs/reports/data_dictionary.csv) already documents field meanings, types, roles, and quality caveats. Read it and check its field coverage. There is no need to rebuild or overwrite this documentation every time the notebook runs.

#### Read and check the reviewed dictionary

The dictionary remains read-only documentation. tolist() checks field order as well as membership; is_unique checks duplicate dictionary entries. The assert statements stop if its coverage does not match the input.

In [42]:
# The reviewed dictionary is documentation, not a table to rebuild on every run.
dictionary = pd.read_csv(dictionary_path, keep_default_na=False)

assert dictionary["field"].tolist() == header
assert dictionary["field"].is_unique
print("Dictionary fields:", len(dictionary))
print("The existing dictionary has not been changed.")

Dictionary fields: 21
The existing dictionary has not been changed.


assert stops execution if a check fails. Confirm that both raw views and the source checksum are unchanged. Phase 2 will create a separate prepared table; this audit only reads and checks data.

#### Confirm source preservation

equals() compares each raw table with its saved copy, including values and types. Recompute the file checksum to check the original bytes are unchanged. These audit cells export no files.

In [43]:
# Confirm that this audit did not change the raw data.
assert raw_df.equals(raw_snapshot)
assert literal_df.equals(literal_snapshot)

current_hash = hashlib.sha256(raw_data_path.read_bytes()).hexdigest()
assert current_hash == source_hash

print("Raw tables and source checksum are unchanged.")
print("No files were exported or cleaned by these audit cells.")

Raw tables and source checksum are unchanged.
No files were exported or cleaned by these audit cells.


### Step 2 review

Review the displayed schema, identifiers, missingness, categories, numeric checks, service consistency, and provisional label baseline together with the analysis log. Source provenance, permitted usage, field units, churn timing, and business grain remain open. No imputations or exclusions have been applied to the source.

### Step 3: Quality review and Phase 2 handoff

**Phase 1 complete (6 November 2026):** See the [quality report](../outputs/reports/data_quality_report.md) for issues DQ01-DQ07, preparation rules, metric definitions, and reconciliation checks. The [analysis log](../outputs/reports/analysis_log.md) records verification and readiness.

Retain all 7,043 unique IDs with valid Yes/No labels (1,869 Yes; 5,174 No). In Phase 2, convert TotalCharges to numeric in a prepared copy, preserving the 11 whitespace-only totals as missing and adding a missing-total flag. Their association with zero tenure does not justify zero imputation. Retain no-service categories. No cleaning has been implemented yet.

Ready for deterministic snapshot preparation, with provenance, usage conditions, business grain, outcome window, and units explicitly unresolved. Keep missing-charge records in churn denominators; report charge coverage separately. Flag segment sizes below 30 as an analyst-selected reporting convention, with uncertainty assessed in Phase 3. All preparation and analysis stay in notebook cells; temporary verification scripts are deleted after checking outputs.

## Phase 2: Data cleaning and analytical preparation

**Planned:** Implement the [Phase 1 preparation rules](../outputs/reports/data_quality_report.md) in this notebook, then define the churn indicator and analytical bands. Reconcile 7,043 IDs, 1,869 Yes labels, 5,174 No labels, zero exclusions, and 11 missing totals before exporting a prepared table.

Keep `raw_df` unchanged and create a separate analytical table. Save generated tables in `outputs/data/` only after the cleaning rules are agreed and documented. Fit learned model preprocessing on training data in Phase 5.


## Phase 3: Exploratory analysis and SQL validation

**Planned:** Answer the README's business questions using contract, tenure, charges, billing, and service comparisons. Show group sizes, churn volumes, and churn rates together.

Add reproducible queries in `sql/`, reconcile their outputs with pandas, and save reviewed charts in `outputs/charts/`. Record evidence and limitations in the analysis log.


## Phase 4: Business interpretation and retention planning

**Planned:** Prioritise segments by evidence, scale, and operational feasibility. Distinguish observed associations from explanations and proposed actions.

Use the recommendation template in the analysis log to define the target segment, action, owner, success measure, and pilot design. Campaign execution requires additional operational data.


## Phase 5: Baseline predictive support

**Planned:** Verify feature timing, split before fitting preprocessing, and compare a dummy classifier, logistic regression, and a simple segmentation rule.

Choose thresholds using training-side validation. Evaluate held-out precision, recall, and lift at a documented contact capacity. Record limitations and any generated artifacts in `outputs/models/`; add model dependencies when implementing this phase.


## Phase 6: Dashboard, reporting, and portfolio delivery

**Planned:** Build the executive overview, segment analysis, and retention priorities pages described in `dashboard/README.md`.

Reconcile dashboard metrics with Python and SQL, write the executive summary, and update the project README with validated findings and artifact links.
